# TrOCR + LiLT/XLM-R baseline — real SER **and** real RE models, scored against Donut

**What changed from the previous version.** Both stages of the pipeline now load models that were
actually fine-tuned on Sinhala and pushed to the Hugging Face Hub — there is no more training inside
this notebook, and no more geometric-nearest-neighbour stand-in for relation extraction:

| Stage | Model | Architecture |
|---|---|---|
| OCR | `danush99/Model_TrOCR-Sin-Handwritten-Text` | standard `VisionEncoderDecoderModel` |
| SER | `danush99/Model_LiLT-SER-IT-SIN` | standard `LiltForTokenClassification` (in stock `transformers`) |
| RE | `danush99/Model_LiLT-RE-JA-SIN` | `LiLTRobertaLikeForRelationExtraction` — **not** in stock `transformers` |

The RE model is the real biaffine relation-extraction head from the LiLT paper's own reference
implementation ([jpWang/LiLT](https://github.com/jpWang/LiLT)), fine-tuned on Sinhala. That
architecture predates the official `transformers` LiLT integration and was never merged into the
library, so Cell 6 below **ports the ~450 lines of model code inline** (config, embeddings,
attention, encoder, the `BiaffineAttention`/`REDecoder` relation head) rather than depending on an
import that does not exist on Kaggle. Concrete incompatibilities between that ~2022-era code and
whatever `transformers` is actually installed were found and patched as they turned up (not
speculative "just in case" fixes — each is a real, observed break):

1. `apply_chunking_to_forward` (the only pruning-era helper this port actually calls — it never
   calls `find_pruneable_heads_and_indices`/`prune_linear_layer`, since head pruning isn't used
   for inference) moved from `transformers.modeling_utils` to `transformers.pytorch_utils` on some
   installs; imported with a `try`/`except` across both locations since a real Kaggle run showed
   `transformers.pytorch_utils` exists there but does NOT export the (unused, and therefore no
   longer imported) pruning helpers.
2. `PreTrainedModel.get_head_mask` was removed entirely on some installs — re-implemented as a tiny
   override (we only ever call it with `head_mask=None`, so this is a two-line fix, not a
   reimplementation of pruning).
3. `get_extended_attention_mask`'s third positional argument changed from `device` to `dtype` on
   some installs — the ported code no longer passes it positionally and lets it default to
   `self.dtype`.

Cell 6 wraps the whole RE stage in a `try`/`except`: if loading this hand-ported architecture fails
for any environment reason, the notebook prints the exact error and continues with Tiers 0-2 rather
than crashing — everything downstream of Cell 6 checks whether the RE model actually loaded.

## Why the published SinFUND numbers are still not comparable to Donut's

| Source | Number | What it actually measures |
|---|---|---|
| Paper, Table 6 | SER F1 = **0.7560** | token classification **given gold OCR text** |
| Paper, Table 6 | RE F1 = **0.3610** | relation extraction **given gold entities** |
| Paper, Table 5 | TrOCR handwritten CER = **0.5253** | OCR, measured separately |
| Donut notebook | pair F1 = **0.0203** | OCR + SER + RE, **from raw pixels** |

## Dataset stats (both approaches, same underlying data)

Both approaches are built from the same raw annotations in `Datasets/SinFund` — Donut flattens them
into `<s_question>...<s_answer>...` tag sequences per document, this notebook keeps them as
words/boxes/entities/pairs — the *shapes* differ but the *counts* must match. Cell 2 below prints:

| Split | Docs | Words | Entities | Q/A pairs |
|---|---|---|---|---|
| train | 80 | 17,362 | 6,484 | 1,520 |
| validation | 20 | 4,047 | 1,435 | 401 |
| **combined** | **100** | — | — | **1,921** |

`donut-doc-understanding.ipynb` Cell 2 reports `train docs=80 val docs=20 q/a pairs=1921` for the
exact same combined count — Cell 2 re-derives this from scratch here and asserts it matches, so a
future change to either construction path cannot silently drift out of sync.

## Tiered evaluation

Each tier swaps one gold component for the real model's output, so you can see exactly where the
cascade loses accuracy. T2b and T3 now use the **real RE model**, not a geometric heuristic.

| Tier | OCR text | Entity labels | Q→A pairing | Isolates |
|---|---|---|---|---|
| **0** | gold | gold | gold linking | harness sanity — must be exactly 1.000 |
| **1** | gold | **LiLT SER** | gold linking | SER error alone (≈ the paper's 0.756 setting) |
| **2** | **TrOCR** | gold | gold linking | **OCR error alone** |
| **2b** | gold | gold | **LiLT RE** | cost of the real RE model, isolated |
| **3** | **TrOCR** | **LiLT SER** | **LiLT RE** | **full pipeline — directly comparable to Donut** |

## One thing this notebook still does in the baseline's favour

**The pipeline is given gold bounding boxes in every tier.** Donut gets only pixels — no layout, no
text detection. A truly end-to-end pipeline would need a detector too, and would score lower. So
Tier 3 is still an **upper bound** on the pipeline, even with a real RE head.

## Before running

* Accelerator **GPU**, Internet **ON**.
* Kaggle secret **`HF_TOKEN`** — **required**, not optional. Unlike
  `donut-doc-understanding.ipynb` (every model it downloads is public),
  `danush99/Model_TrOCR-Sin-Handwritten-Text` here is a **private** Hugging Face repo
  (confirmed: an unauthenticated request to it returns HTTP 401; the two LiLT repos are
  public and don't need a token). Add a secret named exactly `HF_TOKEN` — notebook editor
  -> Add-ons -> Secrets -> Add a new secret, value = a Hugging Face access token with read
  access to that repo — then **toggle it ON for this notebook specifically** (Kaggle does
  not attach an existing secret to a new notebook automatically). Cell 1 now fails fast
  with this exact explanation if the secret isn't attached, rather than running for several
  minutes and crashing confusingly inside Cell 4.
* Attach **`SinFund`** — the original raw dataset (`training_data/`, `testing_data/`, each with
  `annotations/` + `images/`).

In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT, CONFIG, DATASET DISCOVERY
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc, glob, json, random, re
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

TROCR_ID    = "danush99/Model_TrOCR-Sin-Handwritten-Text"
LILT_SER_ID = "danush99/Model_LiLT-SER-IT-SIN"   # standard LiltForTokenClassification
LILT_RE_ID  = "danush99/Model_LiLT-RE-JA-SIN"    # custom LiLTRobertaLikeForRelationExtraction (Cell 6)
LILT_TOK_FALLBACK = "nielsr/lilt-xlm-roberta-base"  # used only if a LiLT repo has no tokenizer files
MAX_SEQ_LENGTH = 512

REPORT = "./pipeline_vs_donut_report.json"

print("=" * 74); print("ENVIRONMENT"); print("=" * 74)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable — enable a GPU accelerator in Kaggle.")
print("GPU:", torch.cuda.get_device_name(0))

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF authentication: configured")
except Exception as e:
    print("Warning: HF_TOKEN unavailable (", e, ")")

# Unlike donut-doc-understanding.ipynb (every model it downloads is public), TROCR_ID
# here is a PRIVATE Hugging Face repo -- confirmed by an unauthenticated request
# returning HTTP 401 (the two LiLT repos returned 200 with no token needed). Fail
# fast HERE, before spending time on dataset discovery/tokenizer setup, rather than
# letting the notebook run for several minutes and then crash confusingly inside
# Cell 4's from_pretrained() with a bare 401.
if HF_TOKEN is None:
    raise RuntimeError(
        f"HF_TOKEN is required: {TROCR_ID!r} is a PRIVATE Hugging Face repo (an "
        "unauthenticated request to it returns HTTP 401; the LiLT SER/RE repos are "
        "public and don't need this). Add a Kaggle secret named exactly 'HF_TOKEN' "
        "(notebook editor -> Add-ons -> Secrets -> Add a new secret) with a Hugging "
        "Face access token that has read access to this repo, then toggle it ON for "
        "THIS notebook specifically (Kaggle does not attach existing secrets to a new "
        "notebook automatically) and re-run.")

def find_sinfund():
    """Search by STRUCTURE, not by a directory name matching literally "SinFund"
    -- Kaggle dataset slugs are lowercased and the folder name actually present
    inside a zip upload can differ from what you'd expect (e.g. the user may
    have zipped the CONTENTS of SinFund rather than the SinFund folder itself,
    so there is no "SinFund"-named directory anywhere in the mount at all).
    Walk every directory under /kaggle/input and '.' and return the first one
    that directly contains a valid training_data/ + testing_data/ pair."""
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if (os.path.isdir(os.path.join(dirpath, "training_data", "annotations"))
                    and os.path.isdir(os.path.join(dirpath, "training_data", "images"))
                    and os.path.isdir(os.path.join(dirpath, "testing_data", "annotations"))
                    and os.path.isdir(os.path.join(dirpath, "testing_data", "images"))):
                return dirpath
    return None

def print_tree(root, max_depth=4):
    """Recursive listing for diagnosing a failed dataset search -- printing only
    the immediate children of /kaggle/input (as the previous version of this
    cell did) hides everything Kaggle nests datasets under."""
    root = os.path.normpath(root)
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        print("  " * depth + "- " + os.path.basename(dirpath) + "/")
        if depth == max_depth:
            for f in filenames[:3]:
                print("  " * (depth + 1) + "- " + f)

# The ORIGINAL raw dataset -- same annotations and same images the Donut
# notebook's references were regenerated from. Using it directly (instead of a
# derived copy) means there is nothing to keep in sync by hand.
SINFUND_ROOT = find_sinfund()
if SINFUND_ROOT is None:
    print("\nCould not find a training_data/ + testing_data/ pair anywhere under "
          "/kaggle/input. Full tree (up to 4 levels deep):")
    if os.path.isdir("/kaggle/input"):
        print_tree("/kaggle/input")
    else:
        print("  /kaggle/input does not exist at all -- no dataset is attached to this notebook.")
    raise FileNotFoundError(
        "Need the raw SinFund dataset attached: a folder containing training_data/ "
        "and testing_data/, each with annotations/ and images/ inside. Check Notebook "
        "Settings -> Input -> Add Input -> Datasets, and check the tree printed above "
        "for the actual path/name Kaggle mounted it under.")

print("\nSinFund:", SINFUND_ROOT)
print("TrOCR   :", TROCR_ID)
print("LiLT SER:", LILT_SER_ID)
print("LiLT RE :", LILT_RE_ID)
print("=" * 74)

ENVIRONMENT
PyTorch: 2.10.0+cu128 | CUDA: 12.8 | available: True
GPU: Tesla T4
HF authentication: configured

SinFund: /kaggle/input/datasets/danushamsc25/sinfund/dataset
TrOCR   : danush99/Model_TrOCR-Sin-Handwritten-Text
LiLT SER: danush99/Model_LiLT-SER-IT-SIN
LiLT RE : danush99/Model_LiLT-RE-JA-SIN


In [2]:
# ============================================================
# CELL 2 — DATA: build train + validation directly from raw SinFund, print stats
# ============================================================
# Same word/box/linking/reading-order/text-cleaning logic as
# regenerate_donut_dataset.py and prepare_lilt_dataset.py, kept in sync by
# hand (Kaggle notebooks can't import local project files). Only the
# validation split is actually used for evaluation below -- the training
# split is built here ONLY to print comparable dataset stats; nothing in this
# notebook trains on it.
from collections import defaultdict, Counter

print("=" * 74); print("DATA"); print("=" * 74)

ROW_BAND = 40

def clean_text(s):
    return re.sub(r"\s+", " ", str(s)).strip()

def norm_box(b):
    x0, y0, x1, y1 = b
    return [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)]

def reading_key(box):
    b = norm_box(box)
    return (round(((b[1] + b[3]) / 2) / ROW_BAND), (b[0] + b[2]) / 2)

def build_split(split_dir):
    ann_dir = os.path.join(SINFUND_ROOT, split_dir, "annotations")
    docs = []
    for fname in sorted(os.listdir(ann_dir)):
        if not fname.endswith(".json"):
            continue
        stem = os.path.splitext(fname)[0]
        data = json.load(open(os.path.join(ann_dir, fname), encoding="utf-8"))
        items = {it["id"]: it for it in data.get("form", [])}

        ordered = sorted(((reading_key(it["box"]), it) for it in data.get("form", [])
                           if clean_text(it.get("text"))), key=lambda e: e[0])

        words, boxes, entity_list = [], [], []
        for _, it in ordered:
            lab = it["label"].upper()
            lab = lab if lab in ("HEADER", "QUESTION", "ANSWER") else "O"
            ws = [w for w in (it.get("words") or []) if clean_text(w.get("text"))]
            if ws:
                for w in ws:
                    words.append(clean_text(w["text"])); boxes.append(norm_box(w["box"]))
            else:
                for t in clean_text(it.get("text")).split():
                    words.append(t); boxes.append(norm_box(it["box"]))
            # Entity-level record. TrOCR was trained on entity-sized crops
            # (mean ~19 chars), so running OCR per ENTITY rather than per word
            # keeps the baseline in its own training distribution -- the
            # favourable choice for it.
            entity_list.append({"text": clean_text(it.get("text")),
                                "box": norm_box(it["box"]), "label": lab})

        q_to_a = defaultdict(list)
        for it in data.get("form", []):
            for link in it.get("linking", []) or []:
                if len(link) != 2:
                    continue
                a, b = link
                ia, ib = items.get(a), items.get(b)
                if not ia or not ib:
                    continue
                if ia["label"] == "question" and ib["label"] == "answer":
                    q_to_a[a].append(b)
                elif ib["label"] == "question" and ia["label"] == "answer":
                    q_to_a[b].append(a)

        pairs = []
        for qid, aids in q_to_a.items():
            q = items[qid]
            qt = clean_text(q.get("text"))
            if not qt:
                continue
            seen, uniq = set(), []
            for aid in aids:
                if aid not in seen:
                    seen.add(aid); uniq.append(aid)
            uniq.sort(key=lambda i: reading_key(items[i]["box"]))
            for aid in uniq:
                at = clean_text(items[aid].get("text"))
                if not at:
                    continue
                pairs.append({"question": qt, "answer": at,
                              "q_box": norm_box(q["box"]),
                              "a_box": norm_box(items[aid]["box"])})
        pairs.sort(key=lambda p: reading_key(p["q_box"]))

        if not words or not pairs:
            continue
        docs.append({"stem": stem, "words": words, "boxes": boxes,
                     "entities": entity_list, "pairs": pairs})
    return docs

lilt_train = build_split("training_data")
lilt_val   = build_split("testing_data")

def image_path(stem, split_dir):
    return os.path.join(SINFUND_ROOT, split_dir, "images", stem + ".png")

REF_DOCS = [[(p["question"], p["answer"]) for p in d["pairs"]] for d in lilt_val]

def split_stats(name, docs):
    n_words = sum(len(d["words"]) for d in docs)
    n_ent = sum(len(d["entities"]) for d in docs)
    n_pairs = sum(len(d["pairs"]) for d in docs)
    labels = Counter(e["label"] for d in docs for e in d["entities"])
    print(f"  {name:12s} docs={len(docs):3d}  words={n_words:6d}  entities={n_ent:5d}  "
          f"pairs={n_pairs:5d}  labels={dict(labels)}")
    return n_pairs

print("\nThis notebook's dataset (built directly from raw SinFund):")
train_pairs = split_stats("train", lilt_train)
val_pairs   = split_stats("validation", lilt_val)
combined = train_pairs + val_pairs
print(f"\ncombined train+validation q/a pairs: {combined}")
print("donut-doc-understanding.ipynb Cell 2 reports: train docs=80 val docs=20 q/a pairs=1921")
print("(same underlying raw annotations, reshaped differently for each approach)")
assert len(lilt_train) == 80 and len(lilt_val) == 20 and combined == 1921, (
    "dataset stats no longer match the Donut notebook's reported counts -- "
    "investigate before trusting any comparison.")
print("MATCH CONFIRMED: both approaches see identical underlying data.")
assert len(lilt_val) > 0 and sum(len(r) for r in REF_DOCS) > 0
print("=" * 74)

DATA

This notebook's dataset (built directly from raw SinFund):
  train        docs= 80  words= 17362  entities= 6484  pairs= 1520  labels={'O': 3070, 'HEADER': 237, 'QUESTION': 1628, 'ANSWER': 1549}
  validation   docs= 20  words=  4047  entities= 1435  pairs=  401  labels={'HEADER': 41, 'O': 623, 'ANSWER': 422, 'QUESTION': 349}

combined train+validation q/a pairs: 1921
donut-doc-understanding.ipynb Cell 2 reports: train docs=80 val docs=20 q/a pairs=1921
(same underlying raw annotations, reshaped differently for each approach)
MATCH CONFIRMED: both approaches see identical underlying data.


In [3]:
# ============================================================
# CELL 3 — METRIC (copied verbatim from the Donut notebook)
# ============================================================
# This is the whole point of the notebook: the SAME scoring function, so the
# numbers are directly comparable. Do not "improve" it here -- any change must
# be mirrored in donut-doc-understanding.ipynb or the comparison silently breaks.
print("=" * 74); print("METRIC"); print("=" * 74)

def levenshtein(a, b):
    if a == b:
        return 0
    if not a or not b:
        return len(a) or len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[len(b)]

def norm_edit_sim(a, b):
    return 1.0 - levenshtein(a, b) / max(len(a), len(b), 1)

def score_pairs(pred_docs, ref_docs):
    """pred_docs / ref_docs: list of [(question, answer), ...] per document.
    Mirrors score_predictions() in the Donut notebook exactly."""
    matched = p_tot = r_tot = exact = 0
    pair_sims = []
    for pp, rp in zip(pred_docs, ref_docs):
        pc, rc = Counter(pp), Counter(rp)
        matched += sum((pc & rc).values()); p_tot += len(pp); r_tot += len(rp)
        exact += (pc == rc)
        p_str = "\n".join(f"{q}: {a}" for q, a in sorted(pp))
        r_str = "\n".join(f"{q}: {a}" for q, a in sorted(rp))
        pair_sims.append(norm_edit_sim(p_str, r_str))
    prec = matched / p_tot if p_tot else 0.0
    rec  = matched / r_tot if r_tot else 0.0
    return {"precision": prec, "recall": rec,
            "f1": 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0,
            "exact_match": exact / max(len(pred_docs), 1),
            "edit_similarity": float(np.mean(pair_sims)) if pair_sims else 0.0}
    # NOTE: no `raw_similarity` here, deliberately. In the Donut notebook that
    # metric is edit similarity over the RAW GENERATED STRING, which still
    # contains the <s_question>/<s_answer>/<sep/> tags -- those tags are always
    # emitted correctly and inflate the score. The pipeline never produces such a
    # string, so any number computed here would not measure the same thing.
    # Compare the two systems on f1 / exact_match / edit_similarity only.

def cer(pred, ref):
    return levenshtein(pred, ref) / max(len(ref), 1)

print("scoring function ready; reference documents:", len(REF_DOCS))

# Tier 0 sanity: perfect input must score exactly 1.0, or the harness is wrong.
t0 = score_pairs(REF_DOCS, REF_DOCS)
print("TIER 0 (gold everything):", {k: round(v, 4) for k, v in t0.items()})
assert abs(t0["f1"] - 1.0) < 1e-9, "harness broken — gold input must score 1.0"
print("=" * 74)

METRIC
scoring function ready; reference documents: 20
TIER 0 (gold everything): {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'exact_match': 1.0, 'edit_similarity': 1.0}


In [4]:
# ============================================================
# CELL 4 — TrOCR: read every entity crop (the OCR stage)
# ============================================================
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("=" * 74); print("TrOCR (OCR STAGE) -- loading saved model from the Hub"); print("=" * 74)

# use_fast=False: recent `transformers` loads ViTImageProcessor as the new "fast"
# implementation by default even for a checkpoint saved with the slow one (the
# notebook's own printed warning says this explicitly: "This is a breaking change
# and may produce slightly different outputs"). For a vision-encoder-decoder model
# fine-tuned against the slow processor's exact resize/normalize behaviour, that
# difference can be large enough to degrade generation severely -- pin the slow
# processor so preprocessing matches what the model was actually trained on.
trocr_proc  = TrOCRProcessor.from_pretrained(TROCR_ID, token=HF_TOKEN, use_fast=False)
trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_ID, token=HF_TOKEN).cuda().eval()
print("loaded", TROCR_ID)

@torch.no_grad()
def ocr_batch(crops, bs=16):
    out = []
    for i in range(0, len(crops), bs):
        chunk = crops[i:i + bs]
        px = trocr_proc(images=chunk, return_tensors="pt").pixel_values.cuda()
        ids = trocr_model.generate(px, max_length=64, num_beams=1)
        out += trocr_proc.batch_decode(ids, skip_special_tokens=True)
    return out

def run_ocr(docs, split_dir):
    """Replace every entity's gold text with TrOCR's reading of its crop."""
    total_cer, n, n_empty = 0.0, 0, 0
    for k, d in enumerate(docs):
        img = Image.open(image_path(d["stem"], split_dir)).convert("RGB")
        W, H = img.size
        crops, idxs = [], []
        for j, e in enumerate(d["entities"]):
            x0, y0, x1, y1 = e["box"]
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(W, x1), min(H, y1)
            if x1 - x0 < 4 or y1 - y0 < 4:
                continue
            crops.append(img.crop((x0, y0, x1, y1))); idxs.append(j)
        texts = ocr_batch(crops) if crops else []
        for j, t in zip(idxs, texts):
            gold = d["entities"][j]["text"]
            d["entities"][j]["ocr_text"] = " ".join(str(t).split()).strip()
            total_cer += cer(d["entities"][j]["ocr_text"], gold); n += 1
            n_empty += (d["entities"][j]["ocr_text"] == "")
        for e in d["entities"]:
            e.setdefault("ocr_text", "")
        if (k + 1) % 5 == 0:
            print(f"  {k+1}/{len(docs)} documents  running CER={total_cer/max(n,1):.4f}  "
                  f"empty_reads={n_empty}/{n}")
    # A CER pinned at exactly 1.0000 with zero variance across hundreds of crops
    # is not "the model is weak" -- it means every single read came back empty,
    # which points at a preprocessing/generation bug, not an OCR quality problem.
    if n and n_empty / n > 0.5:
        print(f"\nWARNING: {n_empty}/{n} entities produced an EMPTY OCR read "
              f"({100*n_empty/n:.1f}%). A uniform empty output across many crops "
              "usually means the image preprocessing or generate() call is broken, "
              "not that the model is simply inaccurate -- check the image processor "
              "(fast vs slow, see the use_fast=False note above) before trusting "
              "this run's OCR-dependent tiers.")
    return total_cer / max(n, 1)

VAL_CER = run_ocr(lilt_val, "testing_data")
print(f"\nentity-level CER on the 20 validation forms: {VAL_CER:.4f}")
print("(paper Table 5 reports 0.2246 printed / 0.5253 handwritten, measured separately)")

# A pair can only match exactly if BOTH strings are read exactly.
ex_q = ex_a = tot = 0
for d in lilt_val:
    for e in d["entities"]:
        if e["label"] == "QUESTION":
            ex_q += (e["ocr_text"] == e["text"])
        elif e["label"] == "ANSWER":
            ex_a += (e["ocr_text"] == e["text"]); tot += 1
nq = sum(1 for d in lilt_val for e in d["entities"] if e["label"] == "QUESTION")
print(f"entities read EXACTLY right: questions {ex_q}/{nq}, answers {ex_a}/{tot}")
del trocr_model; gc.collect(); torch.cuda.empty_cache()
print("=" * 74)

TrOCR (OCR STAGE) -- loading saved model from the Hub


preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/963M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

loaded danush99/Model_TrOCR-Sin-Handwritten-Text


The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  5/20 documents  running CER=1.0000  empty_reads=409/409
  10/20 documents  running CER=1.0000  empty_reads=805/805
  15/20 documents  running CER=1.0000  empty_reads=1096/1096
  20/20 documents  running CER=1.0000  empty_reads=1418/1418


entity-level CER on the 20 validation forms: 1.0000
(paper Table 5 reports 0.2246 printed / 0.5253 handwritten, measured separately)
entities read EXACTLY right: questions 0/349, answers 0/422


In [5]:
# ============================================================
# CELL 5 — LiLT: semantic entity recognition (the SER stage)
# ============================================================
# Inference only -- no training in this notebook. The label scheme (id2label)
# comes from the pushed model's OWN config, not a hardcoded list.
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoTokenizer, LiltForTokenClassification

print("=" * 74); print("LiLT (SER STAGE) -- loading saved model from the Hub"); print("=" * 74)

lilt_ser = LiltForTokenClassification.from_pretrained(LILT_SER_ID, token=HF_TOKEN).cuda().eval()
try:
    lilt_tok = AutoTokenizer.from_pretrained(LILT_SER_ID, token=HF_TOKEN, max_length=MAX_SEQ_LENGTH)
except Exception as e:
    print(f"no tokenizer at {LILT_SER_ID} ({e}); falling back to {LILT_TOK_FALLBACK}")
    lilt_tok = AutoTokenizer.from_pretrained(LILT_TOK_FALLBACK, max_length=MAX_SEQ_LENGTH)

id2label = {int(k): v for k, v in lilt_ser.config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}
print("loaded", LILT_SER_ID, "| label scheme:", sorted(id2label.values()))

# predict_labels() below collapses word-level predictions onto entities by
# stripping a "B-"/"I-" prefix. If the pushed model doesn't use that naming,
# that collapse silently produces nonsense -- fail loudly instead.
assert {"O", "B-QUESTION", "I-QUESTION", "B-ANSWER", "I-ANSWER"} <= set(id2label.values()), (
    f"LILT_SER_ID={LILT_SER_ID} has an unexpected label scheme {sorted(id2label.values())}; "
    "expected IOB tags including B-/I-QUESTION and B-/I-ANSWER.")

def normalize_bbox(b, w, h):
    return [int(1000 * b[0] / w), int(1000 * b[1] / h),
            int(1000 * b[2] / w), int(1000 * b[3] / h)]

class LiltDataset(TorchDataset):
    """Same encoding as the reference LiLT notebook. Inference only: no
    labels are produced or consumed."""
    def __init__(self, docs, split_dir, use_ocr=False):
        self.docs, self.split_dir, self.use_ocr = docs, split_dir, use_ocr
    def __len__(self):
        return len(self.docs)
    def __getitem__(self, i):
        d = self.docs[i]
        with Image.open(image_path(d["stem"], self.split_dir)) as im:
            W, H = im.size
        words, boxes = self._sequence(d)
        bbox = []
        for wtext, box in zip(words, boxes):
            nb = normalize_bbox(box, W, H)
            n = max(1, len(lilt_tok.tokenize(wtext)))
            bbox += [nb] * n
        enc = lilt_tok(" ".join(words), truncation=True, max_length=MAX_SEQ_LENGTH)
        L = len(enc["input_ids"])
        bbox = ([[0, 0, 0, 0]] + bbox + [[0, 0, 0, 0]])[:L]
        bbox += [[0, 0, 0, 0]] * (L - len(bbox))
        enc["bbox"] = bbox
        return {k: torch.tensor(v) for k, v in enc.items()}
    def _sequence(self, d):
        if not self.use_ocr:
            return d["words"], d["boxes"]
        words, boxes = [], []
        for e in d["entities"]:
            for t in (e.get("ocr_text") or "").split():
                words.append(t); boxes.append(e["box"])
        return (words or ["."], boxes or [[0, 0, 1, 1]])

@torch.no_grad()
def predict_labels(docs, split_dir, use_ocr):
    """Return, per document, a predicted label for each ENTITY (majority vote
    over that entity's word tokens)."""
    ds = LiltDataset(docs, split_dir, use_ocr=use_ocr)
    preds = []
    for i, d in enumerate(docs):
        batch = {k: v.unsqueeze(0).cuda() for k, v in ds[i].items()}
        logits = lilt_ser(**batch).logits[0]
        ids = logits.argmax(-1).tolist()
        words, boxes = ds._sequence(d)
        pos, wpred = 1, []
        for wtext in words:
            n = max(1, len(lilt_tok.tokenize(wtext)))
            wpred.append(id2label[ids[pos]] if pos < len(ids) else "O")
            pos += n
        ent_labels, wi = [], 0
        for e in d["entities"]:
            toks = ((e.get("ocr_text") or "").split() if use_ocr else e["text"].split())
            n = max(1, len(toks))
            chunk = wpred[wi:wi + n]; wi += n
            base = [c.split("-")[-1] for c in chunk if c != "O"]
            ent_labels.append(Counter(base).most_common(1)[0][0] if base else "O")
        preds.append(ent_labels)
    return preds

SER_GOLD_TEXT = predict_labels(lilt_val, "testing_data", use_ocr=False)
SER_OCR_TEXT  = predict_labels(lilt_val, "testing_data", use_ocr=True)

def ser_f1(pred_docs):
    tp = fp = fn = 0
    for d, pl in zip(lilt_val, pred_docs):
        for e, p in zip(d["entities"], pl):
            g = e["label"]
            if g != "O" and p == g: tp += 1
            elif p != "O" and p != g: fp += 1
            if g != "O" and p != g: fn += 1
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    return P, R, (2 * P * R / (P + R) if P + R else 0.0)

print(f"\nentity-level SER  (gold text) P/R/F1 = {tuple(round(x,4) for x in ser_f1(SER_GOLD_TEXT))}")
print(f"entity-level SER  (TrOCR text) P/R/F1 = {tuple(round(x,4) for x in ser_f1(SER_OCR_TEXT))}")
print("(paper Table 6 reports token-level SER F1 = 0.7560 on gold text)")
del lilt_ser; gc.collect(); torch.cuda.empty_cache()
print("=" * 74)

LiLT (SER STAGE) -- loading saved model from the Hub


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

loaded danush99/Model_LiLT-SER-IT-SIN | label scheme: ['B-ANSWER', 'B-HEADER', 'B-QUESTION', 'I-ANSWER', 'I-HEADER', 'I-QUESTION', 'O']

entity-level SER  (gold text) P/R/F1 = (0.5968, 0.7254, 0.6548)
entity-level SER  (TrOCR text) P/R/F1 = (0.0, 0.0, 0.0)
(paper Table 6 reports token-level SER F1 = 0.7560 on gold text)


In [6]:
# ============================================================
# CELL 6 — LiLT: relation extraction (the RE stage, REAL biaffine head)
# ============================================================
# LILT_RE_ID uses `LiLTRobertaLikeForRelationExtraction` -- the reference LiLT
# paper's own RE architecture (github.com/jpWang/LiLT), never merged into stock
# `transformers`. This cell ports the needed classes inline (Kaggle notebooks
# can't import local project files, and this is a one-off ~2022 architecture,
# not something pip-installable). See the markdown cell above for the three
# concrete compatibility fixes applied against the current `transformers`.
#
# Everything below is wrapped in try/except: if this hand-ported architecture
# fails to load for any environment reason, RE_MODEL_OK is set False and Cell 7
# reports Tiers 0-2 only rather than crashing the whole notebook.
print("=" * 74); print("LiLT (RE STAGE) -- loading real biaffine head from the Hub"); print("=" * 74)

RE_MODEL_OK = False
try:
    import math as _math
    import torch.nn as nn
    from torch.nn import CrossEntropyLoss
    from transformers import XLMRobertaConfig
    from transformers.activations import ACT2FN
    from transformers.modeling_outputs import (
        BaseModelOutputWithPastAndCrossAttentions,
        BaseModelOutputWithPoolingAndCrossAttentions,
    )
    from transformers.modeling_utils import PreTrainedModel
    # Fix 1/3: apply_chunking_to_forward moved from transformers.modeling_utils to
    # transformers.pytorch_utils at some point after this architecture was written.
    # (find_pruneable_heads_and_indices / prune_linear_layer were part of the same
    # move, but this port never calls them -- head pruning isn't used for
    # inference -- so they are deliberately NOT imported here at all: a real
    # Kaggle run showed transformers.pytorch_utils exists but does NOT export
    # find_pruneable_heads_and_indices on that image, which crashed this whole
    # try block over a function that was never actually needed. Import only
    # what is used, and try both locations for that one function since its exact
    # module has now been observed to differ across environments.)
    try:
        from transformers.pytorch_utils import apply_chunking_to_forward
    except ImportError:
        from transformers.modeling_utils import apply_chunking_to_forward
    from transformers.utils import logging as _logging
    from dataclasses import dataclass
    from typing import Dict, Optional, Tuple
    from transformers.file_utils import ModelOutput

    _logger = _logging.get_logger(__name__)

    class LiLTRobertaLikeConfig(XLMRobertaConfig):
        """Ported from configuration_LiLTRobertaLike.py, multilingual branch only
        (our checkpoints are XLM-R based: vocab_size 250002)."""
        model_type = "liltrobertalike"
        def __init__(self, channel_shrink_ratio=4, max_2d_position_embeddings=1024, **kwargs):
            super().__init__(**kwargs)
            self.channel_shrink_ratio = channel_shrink_ratio
            self.max_2d_position_embeddings = max_2d_position_embeddings

    def _create_position_ids_from_input_ids(input_ids, padding_idx, past_key_values_length=0):
        mask = input_ids.ne(padding_idx).int()
        incremental_indices = (torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask
        return incremental_indices.long() + padding_idx

    class LiLTRobertaLikeTextEmbeddings(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
            self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)
            self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
            self.dropout = nn.Dropout(config.hidden_dropout_prob)
            self.register_buffer("position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)))
            self.position_embedding_type = getattr(config, "position_embedding_type", "absolute")
            self.padding_idx = config.pad_token_id
            self.position_embeddings = nn.Embedding(
                config.max_position_embeddings, config.hidden_size, padding_idx=self.padding_idx)

        def forward(self, input_ids=None, token_type_ids=None, position_ids=None,
                    inputs_embeds=None, past_key_values_length=0):
            if position_ids is None:
                if input_ids is not None:
                    position_ids = _create_position_ids_from_input_ids(
                        input_ids, self.padding_idx, past_key_values_length).to(input_ids.device)
                else:
                    position_ids = self.create_position_ids_from_inputs_embeds(inputs_embeds)
            input_shape = input_ids.size() if input_ids is not None else inputs_embeds.size()[:-1]
            if token_type_ids is None:
                token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=self.position_ids.device)
            if inputs_embeds is None:
                inputs_embeds = self.word_embeddings(input_ids)
            token_type_embeddings = self.token_type_embeddings(token_type_ids)
            embeddings = inputs_embeds + token_type_embeddings
            if self.position_embedding_type == "absolute":
                embeddings = embeddings + self.position_embeddings(position_ids)
            embeddings = self.LayerNorm(embeddings)
            embeddings = self.dropout(embeddings)
            return embeddings, position_ids

        def create_position_ids_from_inputs_embeds(self, inputs_embeds):
            input_shape = inputs_embeds.size()[:-1]
            sequence_length = input_shape[1]
            position_ids = torch.arange(self.padding_idx + 1, sequence_length + self.padding_idx + 1,
                                        dtype=torch.long, device=inputs_embeds.device)
            return position_ids.unsqueeze(0).expand(input_shape)

    class LiLTRobertaLikeLayoutEmbeddings(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.x_position_embeddings = nn.Embedding(config.max_2d_position_embeddings, config.hidden_size // 6)
            self.y_position_embeddings = nn.Embedding(config.max_2d_position_embeddings, config.hidden_size // 6)
            self.h_position_embeddings = nn.Embedding(config.max_2d_position_embeddings, config.hidden_size // 6)
            self.w_position_embeddings = nn.Embedding(config.max_2d_position_embeddings, config.hidden_size // 6)
            self.padding_idx = config.pad_token_id
            self.box_position_embeddings = nn.Embedding(
                config.max_position_embeddings, config.hidden_size // config.channel_shrink_ratio,
                padding_idx=self.padding_idx)
            self.box_linear_embeddings = nn.Linear(config.hidden_size, config.hidden_size // config.channel_shrink_ratio)
            self.LayerNorm = nn.LayerNorm(config.hidden_size // config.channel_shrink_ratio, eps=config.layer_norm_eps)
            self.dropout = nn.Dropout(config.hidden_dropout_prob)

        def forward(self, bbox=None, position_ids=None):
            left  = self.x_position_embeddings(bbox[:, :, 0])
            upper = self.y_position_embeddings(bbox[:, :, 1])
            right = self.x_position_embeddings(bbox[:, :, 2])
            lower = self.y_position_embeddings(bbox[:, :, 3])
            h = self.h_position_embeddings(bbox[:, :, 3] - bbox[:, :, 1])
            w = self.w_position_embeddings(bbox[:, :, 2] - bbox[:, :, 0])
            spatial = torch.cat([left, upper, right, lower, h, w], dim=-1)
            spatial = self.box_linear_embeddings(spatial)
            spatial = spatial + self.box_position_embeddings(position_ids)
            spatial = self.LayerNorm(spatial)
            spatial = self.dropout(spatial)
            return spatial

    class LiLTRobertaLikeSelfAttention(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.num_attention_heads = config.num_attention_heads
            self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
            self.all_head_size = self.num_attention_heads * self.attention_head_size
            self.query = nn.Linear(config.hidden_size, self.all_head_size)
            self.key = nn.Linear(config.hidden_size, self.all_head_size)
            self.value = nn.Linear(config.hidden_size, self.all_head_size)
            r = config.channel_shrink_ratio
            self.layout_query = nn.Linear(config.hidden_size // r, self.all_head_size // r)
            self.layout_key = nn.Linear(config.hidden_size // r, self.all_head_size // r)
            self.layout_value = nn.Linear(config.hidden_size // r, self.all_head_size // r)
            self.dropout = nn.Dropout(config.attention_probs_dropout_prob)
            self.position_embedding_type = getattr(config, "position_embedding_type", "absolute")
            self.is_decoder = config.is_decoder
            self.channel_shrink_ratio = r

        def transpose_for_scores(self, x, r=1):
            new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size // r)
            return x.view(*new_x_shape).permute(0, 2, 1, 3)

        def forward(self, hidden_states, layout_inputs, attention_mask=None, head_mask=None,
                    encoder_hidden_states=None, encoder_attention_mask=None,
                    past_key_value=None, output_attentions=False):
            r = self.channel_shrink_ratio
            layout_value_layer = self.transpose_for_scores(self.layout_value(layout_inputs), r=r)
            layout_key_layer = self.transpose_for_scores(self.layout_key(layout_inputs), r=r)
            layout_query_layer = self.transpose_for_scores(self.layout_query(layout_inputs), r=r)
            mixed_query_layer = self.query(hidden_states)
            key_layer = self.transpose_for_scores(self.key(hidden_states))
            value_layer = self.transpose_for_scores(self.value(hidden_states))
            query_layer = self.transpose_for_scores(mixed_query_layer)

            attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
            layout_attention_scores = torch.matmul(layout_query_layer, layout_key_layer.transpose(-1, -2))

            tmp_attention_scores = attention_scores / _math.sqrt(self.attention_head_size)
            tmp_layout_attention_scores = layout_attention_scores / _math.sqrt(self.attention_head_size // r)
            attention_scores = tmp_attention_scores + tmp_layout_attention_scores
            layout_attention_scores = tmp_layout_attention_scores + tmp_attention_scores

            if attention_mask is not None:
                layout_attention_scores = layout_attention_scores + attention_mask
            layout_attention_probs = self.dropout(nn.Softmax(dim=-1)(layout_attention_scores))
            layout_context_layer = torch.matmul(layout_attention_probs, layout_value_layer)
            layout_context_layer = layout_context_layer.permute(0, 2, 1, 3).contiguous()
            new_shape = layout_context_layer.size()[:-2] + (self.all_head_size // r,)
            layout_context_layer = layout_context_layer.view(*new_shape)

            if attention_mask is not None:
                attention_scores = attention_scores + attention_mask
            attention_probs = self.dropout(nn.Softmax(dim=-1)(attention_scores))
            context_layer = torch.matmul(attention_probs, value_layer)
            context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
            new_shape = context_layer.size()[:-2] + (self.all_head_size,)
            context_layer = context_layer.view(*new_shape)
            return ((context_layer, layout_context_layer),)

    class LiLTRobertaLikeSelfOutput(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.hidden_size, config.hidden_size)
            self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
            self.dropout = nn.Dropout(config.hidden_dropout_prob)
        def forward(self, hidden_states, input_tensor):
            hidden_states = self.dropout(self.dense(hidden_states))
            return self.LayerNorm(hidden_states + input_tensor)

    class LiLTRobertaLikeAttention(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.self = LiLTRobertaLikeSelfAttention(config)
            self.output = LiLTRobertaLikeSelfOutput(config)
            ori = config.hidden_size
            config.hidden_size = config.hidden_size // config.channel_shrink_ratio
            self.layout_output = LiLTRobertaLikeSelfOutput(config)
            config.hidden_size = ori
        def forward(self, hidden_states, layout_inputs, attention_mask=None, head_mask=None,
                    encoder_hidden_states=None, encoder_attention_mask=None,
                    past_key_value=None, output_attentions=False):
            self_outputs = self.self(hidden_states, layout_inputs, attention_mask, head_mask,
                                     encoder_hidden_states, encoder_attention_mask,
                                     past_key_value, output_attentions)
            attention_output = self.output(self_outputs[0][0], hidden_states)
            layout_attention_output = self.layout_output(self_outputs[0][1], layout_inputs)
            return ((attention_output, layout_attention_output),) + self_outputs[1:]

    class LiLTRobertaLikeIntermediate(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
            self.intermediate_act_fn = ACT2FN[config.hidden_act] if isinstance(config.hidden_act, str) else config.hidden_act
        def forward(self, hidden_states):
            return self.intermediate_act_fn(self.dense(hidden_states))

    class LiLTRobertaLikeOutput(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
            self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
            self.dropout = nn.Dropout(config.hidden_dropout_prob)
        def forward(self, hidden_states, input_tensor):
            hidden_states = self.dropout(self.dense(hidden_states))
            return self.LayerNorm(hidden_states + input_tensor)

    class LiLTRobertaLikeLayer(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.chunk_size_feed_forward = config.chunk_size_feed_forward
            self.seq_len_dim = 1
            self.attention = LiLTRobertaLikeAttention(config)
            self.is_decoder = config.is_decoder
            self.intermediate = LiLTRobertaLikeIntermediate(config)
            self.output = LiLTRobertaLikeOutput(config)
            ori_h, ori_i = config.hidden_size, config.intermediate_size
            config.hidden_size = config.hidden_size // config.channel_shrink_ratio
            config.intermediate_size = config.intermediate_size // config.channel_shrink_ratio
            self.layout_intermediate = LiLTRobertaLikeIntermediate(config)
            self.layout_output = LiLTRobertaLikeOutput(config)
            config.hidden_size, config.intermediate_size = ori_h, ori_i

        def forward(self, hidden_states, layout_inputs, attention_mask=None, head_mask=None,
                    encoder_hidden_states=None, encoder_attention_mask=None,
                    past_key_value=None, output_attentions=False):
            self_attention_outputs = self.attention(
                hidden_states, layout_inputs, attention_mask, head_mask,
                output_attentions=output_attentions, past_key_value=None)
            attention_output = self_attention_outputs[0][0]
            layout_attention_output = self_attention_outputs[0][1]
            outputs = self_attention_outputs[1:]

            layer_output = apply_chunking_to_forward(
                self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, attention_output)
            layout_layer_output = apply_chunking_to_forward(
                self.layout_feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, layout_attention_output)
            return ((layer_output, layout_layer_output),) + outputs

        def feed_forward_chunk(self, attention_output):
            return self.output(self.intermediate(attention_output), attention_output)
        def layout_feed_forward_chunk(self, attention_output):
            return self.layout_output(self.layout_intermediate(attention_output), attention_output)

    class LiLTRobertaLikeEncoder(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.config = config
            self.layer = nn.ModuleList([LiLTRobertaLikeLayer(config) for _ in range(config.num_hidden_layers)])
        def forward(self, hidden_states, layout_inputs, attention_mask=None, head_mask=None,
                    encoder_hidden_states=None, encoder_attention_mask=None,
                    past_key_values=None, use_cache=None, output_attentions=False,
                    output_hidden_states=False, return_dict=True):
            for i, layer_module in enumerate(self.layer):
                layer_head_mask = head_mask[i] if head_mask is not None else None
                layer_outputs = layer_module(hidden_states, layout_inputs, attention_mask, layer_head_mask,
                                             encoder_hidden_states, encoder_attention_mask, None, output_attentions)
                hidden_states = layer_outputs[0][0]
                layout_inputs = layer_outputs[0][1]
            return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states), layout_inputs

    class LiLTRobertaLikePooler(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.hidden_size, config.hidden_size)
            self.activation = nn.Tanh()
        def forward(self, hidden_states):
            return self.activation(self.dense(hidden_states[:, 0]))

    class LiLTRobertaLikePreTrainedModel(PreTrainedModel):
        config_class = LiLTRobertaLikeConfig
        base_model_prefix = "liltrobertalike"
        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.Embedding):
                module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()
            elif isinstance(module, nn.LayerNorm):
                module.bias.data.zero_()
                module.weight.data.fill_(1.0)
        # Fix 2/3: PreTrainedModel.get_head_mask was removed from current
        # transformers. We only ever call it with head_mask=None, so this
        # two-line replacement is sufficient (not a full reimplementation of
        # head pruning, which this notebook never uses).
        def get_head_mask(self, head_mask, num_hidden_layers):
            if head_mask is not None:
                raise NotImplementedError("head_mask is not supported in this inference-only port")
            return [None] * num_hidden_layers

    class LiLTRobertaLikeModel(LiLTRobertaLikePreTrainedModel):
        def __init__(self, config, add_pooling_layer=True):
            super().__init__(config)
            self.config = config
            self.embeddings = LiLTRobertaLikeTextEmbeddings(config)
            self.layout_embeddings = LiLTRobertaLikeLayoutEmbeddings(config)
            self.encoder = LiLTRobertaLikeEncoder(config)
            self.pooler = LiLTRobertaLikePooler(config) if add_pooling_layer else None
            self.init_weights()

        def get_input_embeddings(self):
            return self.embeddings.word_embeddings
        def set_input_embeddings(self, value):
            self.embeddings.word_embeddings = value

        def forward(self, input_ids=None, bbox=None, attention_mask=None, token_type_ids=None,
                    position_ids=None, head_mask=None, inputs_embeds=None,
                    encoder_hidden_states=None, encoder_attention_mask=None,
                    past_key_values=None, use_cache=None, output_attentions=None,
                    output_hidden_states=None, return_dict=None):
            input_shape = input_ids.size()
            batch_size, seq_length = input_shape
            device = input_ids.device
            if attention_mask is None:
                attention_mask = torch.ones((batch_size, seq_length), device=device)
            if token_type_ids is None:
                token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=device)
            if bbox is None:
                bbox = torch.zeros(tuple(list(input_shape) + [4]), dtype=torch.long, device=device)

            # Fix 3/3: get_extended_attention_mask's 3rd positional arg changed
            # from `device` to `dtype` in current transformers; omit it so it
            # defaults to self.dtype (equivalent to the original behaviour).
            extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)
            head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)

            embedding_output, position_ids = self.embeddings(
                input_ids=input_ids, position_ids=position_ids, token_type_ids=token_type_ids,
                inputs_embeds=inputs_embeds)
            layout_embedding_output = self.layout_embeddings(bbox=bbox, position_ids=position_ids)

            encoder_outputs, layout_encoder_outputs = self.encoder(
                embedding_output, layout_embedding_output, attention_mask=extended_attention_mask,
                head_mask=head_mask, return_dict=True)

            sequence_output = encoder_outputs[0]
            pooled_output = self.pooler(sequence_output) if self.pooler is not None else None
            return BaseModelOutputWithPoolingAndCrossAttentions(
                last_hidden_state=sequence_output, pooler_output=pooled_output), layout_encoder_outputs

    @dataclass
    class ReOutput(ModelOutput):
        loss: Optional[torch.FloatTensor] = None
        logits: torch.FloatTensor = None
        hidden_states: Optional[Tuple[torch.FloatTensor]] = None
        attentions: Optional[Tuple[torch.FloatTensor]] = None
        entities: Optional[Dict] = None
        relations: Optional[Dict] = None
        pred_relations: Optional[Dict] = None

    class BiaffineAttention(nn.Module):
        """Biaffine attention for binary relation classification (End-to-end
        neural relation extraction using deep biaffine attention, arXiv:1812.11275)."""
        def __init__(self, in_features, out_features):
            super().__init__()
            self.bilinear = nn.Bilinear(in_features, in_features, out_features, bias=False)
            self.linear = nn.Linear(2 * in_features, out_features, bias=True)
        def forward(self, x_1, x_2):
            return self.bilinear(x_1, x_2) + self.linear(torch.cat((x_1, x_2), dim=-1))

    class REDecoder(nn.Module):
        def __init__(self, config, input_size):
            super().__init__()
            self.entity_emb = nn.Embedding(3, input_size, scale_grad_by_freq=True)
            projection = nn.Sequential(
                nn.Linear(input_size * 2, config.hidden_size), nn.ReLU(), nn.Dropout(config.hidden_dropout_prob),
                nn.Linear(config.hidden_size, config.hidden_size // 2), nn.ReLU(), nn.Dropout(config.hidden_dropout_prob))
            import copy as _copy
            self.ffnn_head = _copy.deepcopy(projection)
            self.ffnn_tail = _copy.deepcopy(projection)
            self.rel_classifier = BiaffineAttention(config.hidden_size // 2, 2)
            self.loss_fct = CrossEntropyLoss()

        def build_relation(self, relations, entities):
            batch_size = len(relations)
            new_relations = []
            for b in range(batch_size):
                if len(entities[b]["start"]) <= 2:
                    entities[b] = {"end": [1, 1], "label": [0, 0], "start": [0, 0]}
                all_possible_relations = set(
                    (i, j) for i in range(len(entities[b]["label"])) for j in range(len(entities[b]["label"]))
                    if entities[b]["label"][i] == 1 and entities[b]["label"][j] == 2)
                if len(all_possible_relations) == 0:
                    all_possible_relations = set([(0, 1)])
                positive_relations = set(zip(relations[b]["head"], relations[b]["tail"]))
                negative_relations = all_possible_relations - positive_relations
                positive_relations = set(i for i in positive_relations if i in all_possible_relations)
                reordered = list(positive_relations) + list(negative_relations)
                relation_per_doc = {"head": [i[0] for i in reordered], "tail": [i[1] for i in reordered],
                                    "label": [1] * len(positive_relations) + [0] * (len(reordered) - len(positive_relations))}
                new_relations.append(relation_per_doc)
            return new_relations, entities

        def get_predicted_relations(self, logits, relations, entities):
            pred_relations = []
            for i, pred_label in enumerate(logits.argmax(-1)):
                if pred_label != 1:
                    continue
                rel = {"head_id": relations["head"][i], "tail_id": relations["tail"][i]}
                pred_relations.append(rel)
            return pred_relations

        def forward(self, hidden_states, entities, relations):
            device = hidden_states.device
            relations, entities = self.build_relation(relations, entities)
            all_pred_relations = []
            for b in range(hidden_states.size(0)):
                head_entities = torch.tensor(relations[b]["head"], device=device)
                tail_entities = torch.tensor(relations[b]["tail"], device=device)
                entities_start_index = torch.tensor(entities[b]["start"], device=device)
                entities_labels = torch.tensor(entities[b]["label"], device=device)
                head_index = entities_start_index[head_entities]
                head_label_repr = self.entity_emb(entities_labels[head_entities])
                tail_index = entities_start_index[tail_entities]
                tail_label_repr = self.entity_emb(entities_labels[tail_entities])
                head_repr = torch.cat((hidden_states[b][head_index], head_label_repr), dim=-1)
                tail_repr = torch.cat((hidden_states[b][tail_index], tail_label_repr), dim=-1)
                heads = self.ffnn_head(head_repr)
                tails = self.ffnn_tail(tail_repr)
                logits = self.rel_classifier(heads, tails)
                all_pred_relations.append(self.get_predicted_relations(logits, relations[b], entities[b]))
            return None, all_pred_relations

    class LiLTRobertaLikeForRelationExtraction(LiLTRobertaLikePreTrainedModel):
        _keys_to_ignore_on_load_unexpected = [r"pooler"]
        _keys_to_ignore_on_load_missing = [r"position_ids"]
        def __init__(self, config):
            super().__init__(config)
            self.lilt = LiLTRobertaLikeModel(config, add_pooling_layer=False)
            self.dropout = nn.Dropout(config.hidden_dropout_prob)
            self.extractor = REDecoder(config, config.hidden_size + config.hidden_size // config.channel_shrink_ratio)
            self.init_weights()

        def forward(self, input_ids=None, bbox=None, attention_mask=None, entities=None, relations=None):
            outputs, layout_outputs = self.lilt(input_ids, bbox=bbox, attention_mask=attention_mask)
            sequence_output = torch.cat([outputs[0], layout_outputs], -1)
            sequence_output = self.dropout(sequence_output)
            _, pred_relations = self.extractor(sequence_output, entities, relations)
            return ReOutput(pred_relations=pred_relations)

    # Load config explicitly (matches the reference run_xfun_re.py's own pattern
    # of AutoConfig-then-model, rather than relying on from_pretrained's implicit
    # config resolution) and request loading diagnostics: a checkpoint/architecture
    # mismatch here does NOT raise by default -- transformers happily fills any
    # unmatched parameter with its random init and continues, which would silently
    # produce near-random (not obviously wrong) predictions instead of a crash.
    _re_config = LiLTRobertaLikeConfig.from_pretrained(LILT_RE_ID, token=HF_TOKEN)
    re_model, _loading_info = LiLTRobertaLikeForRelationExtraction.from_pretrained(
        LILT_RE_ID, config=_re_config, token=HF_TOKEN, output_loading_info=True)
    _missing = sorted(_loading_info.get("missing_keys", []))
    _unexpected = sorted(_loading_info.get("unexpected_keys", []))
    print(f"loaded {LILT_RE_ID}  missing_keys={len(_missing)}  unexpected_keys={len(_unexpected)}")
    if _missing:
        print("  missing (first 10):", _missing[:10])
    if _unexpected:
        print("  unexpected (first 10):", _unexpected[:10])
    # If the embedding table or the relation-extraction head itself failed to
    # load, every prediction downstream would be meaningless -- fail loudly here
    # (caught by the except below) rather than silently reporting a real-looking
    # but randomly-initialized F1 number.
    _critical = [k for k in _missing if "extractor" in k or "word_embeddings" in k or "layout_embeddings" in k]
    if _critical:
        raise RuntimeError(f"Critical weights missing from checkpoint: {_critical[:5]} -- "
                           "the ported architecture does not match this checkpoint.")
    re_model = re_model.cuda().eval()
    print("RE model ready (LiLTRobertaLikeForRelationExtraction, ported)")
    RE_MODEL_OK = True

except Exception as e:
    import traceback
    print("\nRE MODEL LOAD/DEFINE FAILED -- continuing without Tiers 2b/3's real RE head.")
    print("Everything else in this notebook (Tiers 0-2, all dataset stats) is unaffected.")
    traceback.print_exc()

LABEL2INT = {"HEADER": 0, "QUESTION": 1, "ANSWER": 2}

def build_re_inputs(doc, ent_labels, text_fn, W, H, max_len=512):
    """Tokenize a document's entities in reading order (matches xfun.py's own
    construction: per-entity tokenize with add_special_tokens=False, no BOS/EOS
    -- confirmed by reading run_xfun_re.py, which feeds the dataset's raw
    input_ids straight into the model with no extra special-token insertion).
    Boxes are normalized to the model's 0-1000 layout scale with the SAME
    normalize_bbox() used for the SER stage (entity boxes here are raw pixel
    coordinates, not already normalized).
    Returns (input_ids, bbox, entities_dict, entity_refs) where entity_refs[i]
    is the original entity dict for RE-model entity index i."""
    input_ids, bbox, ent_start, ent_end, ent_label, entity_refs = [], [], [], [], [], []
    for e, lab in zip(doc["entities"], ent_labels):
        txt = text_fn(e)
        if not txt:
            continue
        ids = lilt_tok(txt, add_special_tokens=False)["input_ids"]
        if not ids or len(input_ids) + len(ids) > max_len:
            continue
        start = len(input_ids)
        input_ids.extend(ids)
        nb = normalize_bbox(e["box"], W, H)
        bbox.extend([nb] * len(ids))
        if lab in LABEL2INT:
            ent_start.append(start); ent_end.append(start + len(ids))
            ent_label.append(LABEL2INT[lab]); entity_refs.append(e)
    return input_ids, bbox, {"start": ent_start, "end": ent_end, "label": ent_label}, entity_refs

@torch.no_grad()
def re_model_pairs(doc, ent_labels, use_ocr, split_dir="testing_data"):
    """Real biaffine RE head: enumerate every QUESTION x ANSWER candidate in the
    document and keep the ones the trained model classifies as related."""
    text_fn = (lambda e: (e.get("ocr_text") or "")) if use_ocr else (lambda e: e["text"])
    with Image.open(image_path(doc["stem"], split_dir)) as im:
        W, H = im.size
    input_ids, bbox, entities, entity_refs = build_re_inputs(doc, ent_labels, text_fn, W, H)
    if len(input_ids) == 0 or not entities["start"]:
        return []
    ids_t = torch.tensor([input_ids], dtype=torch.long).cuda()
    bbox_t = torch.tensor([bbox], dtype=torch.long).cuda()
    mask_t = torch.ones_like(ids_t)
    out = re_model(input_ids=ids_t, bbox=bbox_t, attention_mask=mask_t,
                   entities=[entities], relations=[{"head": [], "tail": []}])
    pairs = []
    for rel in out.pred_relations[0]:
        q = entity_refs[rel["head_id"]]; a = entity_refs[rel["tail_id"]]
        pairs.append((text_fn(q), text_fn(a)))
    return pairs

print("RE_MODEL_OK =", RE_MODEL_OK)
print("=" * 74)

LiLT (RE STAGE) -- loading real biaffine head from the Hub


config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

You are using a model of type lilt to instantiate a model of type liltrobertalike. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]


RE MODEL LOAD/DEFINE FAILED -- continuing without Tiers 2b/3's real RE head.
Everything else in this notebook (Tiers 0-2, all dataset stats) is unaffected.
RE_MODEL_OK = False


Traceback (most recent call last):
  File "/tmp/ipykernel_58/2634347331.py", line 482, in <cell line: 0>
    re_model, _loading_info = LiLTRobertaLikeForRelationExtraction.from_pretrained(
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4072, in from_pretrained
    model = cls(config, *model_args, **model_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2634347331.py", line 462, in __init__
    super().__init__(config)
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 1300, in __init__
    self.config._experts_implementation_internal = self._check_and_adjust_experts_implementation(
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 1893, in _check_and_adjust_expe

In [7]:
# ============================================================
# CELL 7 — PAIRING (gold-link tiers) + TIERED RESULTS
# ============================================================
print("=" * 74); print("PAIRING + RESULTS"); print("=" * 74)

def gold_link_pairs(doc, ent_labels, use_ocr):
    """Use the gold question->answer links, but read the TEXT from whichever
    source the tier specifies. Isolates OCR/SER error from pairing error."""
    by_box = {tuple(e["box"]): (e, l) for e, l in zip(doc["entities"], ent_labels)}
    out = []
    for p in doc["pairs"]:
        qe = by_box.get(tuple(p["q_box"])); ae = by_box.get(tuple(p["a_box"]))
        if qe is None or ae is None:
            continue
        if use_ocr:
            out.append(((qe[0].get("ocr_text") or ""), (ae[0].get("ocr_text") or "")))
        else:
            out.append((qe[0]["text"], ae[0]["text"]))
    return out

GOLD_LABELS = [[e["label"] for e in d["entities"]] for d in lilt_val]

tiers = {
 "T0 gold OCR + gold SER + gold link":
    [gold_link_pairs(d, gl, False) for d, gl in zip(lilt_val, GOLD_LABELS)],
 "T1 gold OCR + LiLT SER + gold link":
    [gold_link_pairs(d, pl, False) for d, pl in zip(lilt_val, SER_GOLD_TEXT)],
 "T2 TrOCR    + gold SER + gold link":
    [gold_link_pairs(d, gl, True)  for d, gl in zip(lilt_val, GOLD_LABELS)],
}

if RE_MODEL_OK:
    print("running the real RE model over all validation docs (Tiers 2b, 3)...")
    tiers["T2b gold OCR + gold SER + LiLT RE (RE-model ceiling)"] = \
        [re_model_pairs(d, gl, False) for d, gl in zip(lilt_val, GOLD_LABELS)]
    tiers["T3 TrOCR    + LiLT SER + LiLT RE  <-- comparable to Donut"] = \
        [re_model_pairs(d, pl, True) for d, pl in zip(lilt_val, SER_OCR_TEXT)]
else:
    print("RE_MODEL_OK is False -- Tiers 2b/3 are unavailable this run (see Cell 6's traceback).")

results = {name: score_pairs(pred, REF_DOCS) for name, pred in tiers.items()}

hdr = f"{'tier':<58}{'P':>8}{'R':>8}{'F1':>8}{'exact':>8}{'editsim':>9}"
print("\n" + hdr); print("-" * len(hdr))
for name, m in results.items():
    print(f"{name:<58}{m['precision']:>8.4f}{m['recall']:>8.4f}{m['f1']:>8.4f}"
          f"{m['exact_match']:>8.4f}{m['edit_similarity']:>9.4f}")

if RE_MODEL_OK:
    ceil = results["T2b gold OCR + gold SER + LiLT RE (RE-model ceiling)"]["f1"]
    print(f"\nRE-model ceiling: with PERFECT text and PERFECT labels the real biaffine")
    print(f"RE head reaches F1={ceil:.4f}. Tier 3 cannot exceed this. For reference the")
    print(f"paper's own RE head (trained on gold text/entities, not ported here) reports")
    print(f"F1=0.3610 on the original XFUND-style task (Table 6).")

print("\n" + "=" * 74)
print("COMPARISON — same metric, same 20 documents, same references")
print("=" * 74)
# Donut is NOT executed in this notebook. These numbers are copied from the
# last real Kaggle run of donut-doc-understanding.ipynb -- update this dict if
# you re-run that notebook. raw_similarity (0.5588 there) is deliberately NOT
# listed: it is computed over the raw generated string including structural
# tags, which the pipeline never produces, so it isn't comparable.
DONUT = {"f1": 0.0203, "exact_match": 0.0000, "edit_similarity": 0.2681}
T3_KEY = "T3 TrOCR    + LiLT SER + LiLT RE  <-- comparable to Donut"
if RE_MODEL_OK:
    p3 = results[T3_KEY]
    print(f"  TrOCR+LiLT pipeline (end-to-end) : F1={p3['f1']:.4f}  "
          f"exact={p3['exact_match']:.4f}  editsim={p3['edit_similarity']:.4f}")
    print(f"  Donut (end-to-end, from pixels)  : F1={DONUT['f1']:.4f}  "
          f"exact={DONUT['exact_match']:.4f}  editsim={DONUT['edit_similarity']:.4f}")
    winner = "Donut" if DONUT["f1"] > p3["f1"] else ("pipeline" if p3["f1"] > DONUT["f1"] else "tie")
    print(f"\n  higher end-to-end pair F1: {winner}")
    print("  NOTE: the pipeline was given GOLD BOXES in every tier; Donut was given only")
    print("  pixels. Tier 3 is therefore an UPPER BOUND on the pipeline.")
else:
    p3 = None
    print("  Tier 3 unavailable this run (RE model failed to load) -- cannot compare to Donut.")
    print("  Fix the RE model load (see Cell 6's traceback) and re-run before drawing conclusions.")

json.dump({"entity_cer": VAL_CER,
           "ser_gold_text": ser_f1(SER_GOLD_TEXT),
           "ser_ocr_text": ser_f1(SER_OCR_TEXT),
           "re_model_ok": RE_MODEL_OK,
           "tiers": results, "donut_reference": DONUT},
          open(REPORT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nsaved", REPORT)
print("=" * 74)

PAIRING + RESULTS
RE_MODEL_OK is False -- Tiers 2b/3 are unavailable this run (see Cell 6's traceback).

tier                                                             P       R      F1   exact  editsim
---------------------------------------------------------------------------------------------------
T0 gold OCR + gold SER + gold link                          1.0000  1.0000  1.0000  1.0000   1.0000
T1 gold OCR + LiLT SER + gold link                          1.0000  1.0000  1.0000  1.0000   1.0000
T2 TrOCR    + gold SER + gold link                          0.0000  0.0000  0.0000  0.0000   0.0861

COMPARISON — same metric, same 20 documents, same references
  Tier 3 unavailable this run (RE model failed to load) -- cannot compare to Donut.
  Fix the RE model load (see Cell 6's traceback) and re-run before drawing conclusions.

saved ./pipeline_vs_donut_report.json


In [8]:
# ============================================================
# CELL 8 — QUALITATIVE: where the cascade actually breaks
# ============================================================
print("=" * 74); print("QUALITATIVE"); print("=" * 74)

d  = lilt_val[0]
pl = SER_OCR_TEXT[0]
print(f"document: {d['stem']}\n")
print("OCR read-back on the first 8 entities (gold -> TrOCR):")
for e in d["entities"][:8]:
    ok = "OK " if e["ocr_text"] == e["text"] else "ERR"
    print(f"  [{ok}] {e['label']:<8} {e['text'][:44]!r}")
    print(f"        ->            {e['ocr_text'][:44]!r}")

if RE_MODEL_OK:
    print("\npipeline pairs (Tier 3, real RE model) vs gold, first 5:")
    pred = re_model_pairs(d, pl, True)
    for q, a in pred[:5]:
        print(f"  PRED  {q[:38]!r} -> {a[:30]!r}")
else:
    print("\nTier 3 unavailable this run (RE model failed to load) -- no predicted pairs to show.")
for q, a in REF_DOCS[0][:5]:
    print(f"  REF   {q[:38]!r} -> {a[:30]!r}")

print("\nWhy exact-match pair F1 collapses for the pipeline:")
print(f"  a pair counts only if BOTH strings are read exactly.")
print(f"  entity-level CER = {VAL_CER:.4f}; with ~19 characters per field the")
print(f"  chance of an exact read is roughly (1-VAL_CER)**19 = {max(0.0,1-VAL_CER)**19:.2e}.")
print("  That is the cascading-error argument, measured rather than asserted.")
print("=" * 74)

QUALITATIVE
document: sin_val_0

OCR read-back on the first 8 entities (gold -> TrOCR):
  [ERR] HEADER   'B කොටස'
        ->            ''
  [ERR] O        '15'
        ->            ''
  [ERR] HEADER   'ග්\u200dරාම නිලධාරීගේ නිර්දේශය'
        ->            ''
  [ERR] O        ':'
        ->            ''
  [ERR] ANSWER   'කහටගස්දිගිලිය , සමගි මාවත , අංක 23 නිවසේ'
        ->            ''
  [ERR] O        'හි'
        ->            ''
  [ERR] QUESTION 'පදිංචි'
        ->            ''
  [ERR] ANSWER   'මල්ලව ආරච්චිගේ දොන් කවිදු ඉදුසර'
        ->            ''

Tier 3 unavailable this run (RE model failed to load) -- no predicted pairs to show.
  REF   'පදිංචි' -> 'කහටගස්දිගිලිය , සමගි මාවත , අං'
  REF   'ශිෂ්\u200dයයාව / ශිෂ්\u200dයාව .' -> 'මල්ලව ආරච්චිගේ දොන් කවිදු ඉදුස'
  REF   'ආබාධිත' -> 'පෝලියෝ'
  REF   '.පාසලේ' -> 'කහටගස්දිගිලිය මහසෙන් කණිෂ්ට වි'
  REF   'ශ්\u200dරේණියේ / වසරේ .' -> '7 වන'

Why exact-match pair F1 collapses for the pipeline:
  a pair counts only if BOTH strings 